# Neural Networks - Backpropagation

How a NN can improve itself if it makes a mistake?

## 1. Why Learning Requires Gradients

- If info missing in forward?
- Learning = find $W, b$ -> accurate predictions. -> Modify W, b if error.
- 2 info needed:
  - How wrong (value)
  - Where to move W and b (direction)
- Gradient = How much Loss changes when param changes abit

```math
\^{y} = wx + b \\
L = (\^{y} - y)^2 = (wx + b - y)^2
```

then derivatives: $\frac{\partial L}{\partial w}$ and $\frac{\partial L}{\partial b}$

- Infinitesimal change: $w \rightarrow w + \Delta w$
  - sign (+/-) -> direction
  - magnitude -> sensitive
- Not how to update the params yet.

In [1]:
import numpy as np

x = 5.0
y = 20.0

w = 2.0
b = 1.0


def forward(x, w, b):
    return w * x + b

def mse_loss(y_pred, y_true):
    return (y_pred - y_true)**2

# Perturb
eps = 1e-5

loss_plus = mse_loss(forward(x, w + eps, b), y)
loss_minus = mse_loss(forward(x, w - eps, b), y)

numerical_gradient = (loss_plus - loss_minus) / (2 * eps) # see lesson 11 - Gradient Checking

print("Estimated dL/dw:", numerical_gradient)

Estimated dL/dw: -90.0000000001455


## 2. Computational Graphs*

- *Core* of backprop.
- Don't do one gigantic derivative. -> One node at a time -> local derivative.

```
a ----\
       (+)----u----\
b ----/             (* )----v----(^2)----L
                  /
c ---------------/
```

- Walk through this graph backwards -> compute gradients.
- DAG:
  - Nodes = params, store:
    - `v`: value
    - `f`: operation
    - `parent1`, `parent2`,...: parents
    - `gradient`
  - Edges = Computations

> Why *Acyclic* graphs: Because if a later node in the graph points back to a random previous node, it can't compute that later node before any node before, OBVIOUSLY!

- **Forward** stores values.
- **Backward** stores gradients.
- 2 graph types:
  - **Dynamic graphs**: built as Python code executes, intuitive, easy to debug.
  - **Static graphs**: the graph is defined first, then optimised before execution, aggressive compiler optimisations.

In [3]:
class Node:
    ...
    def add(a, b):
        ...

    def mul(a, b):
        ...

    def square(a):
        ...

In [4]:
# The same one in PyTorch
# It builds a computational graph automatically

import torch

a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = torch.tensor(4.0, requires_grad=True)

L = ((a + b) * c)**2

print(L)
print(L.grad_fn)

tensor(400., grad_fn=<PowBackward0>)


## 3. Local Derivatives

- Each operation only knows its own local rule.
- Each operation deserves its onw class, each has:
  - `forward`
  - `backward`

**Chain rule**:

```math
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial z}.\frac{\partial z}{\partial x}
```

- So backward applies *Chain rule*.

After implementation, verify the result with Numerical differentiation.

```math
\frac{\partial z}{\partial x} \approx \frac{f(x+h) - f(x-h)}{2h}
```

- Pytorch stores values in forward -> memory-computation trade-off. -> **gradient checkpointing** discards and recompute later.

In [ ]:
import numpy as np

class Node:

    def __init__(self, value):
        self.value = np.asarray(value, dtype=float)
        self.grad = np.zeros_like(self.value)

class Add:

    def forward(self, x, y):
        ...

    def backward(self, grad_output):
        return grad_output * 1.0

In [6]:
# Torch comparison

import torch

x = torch.tensor(3.0, requires_grad=True)
y = torch.tensor(4.0, requires_grad=True)

z = x *y
z.backward()

print(x.grad)
print(y.grad)

tensor(4.)
tensor(3.)


## 4. The Chain Rule (The Heart of Deep Learning)

Total loss:

```math
L(x) = f_5(f_4(f_3(f_2(f_1))))
```

- Loss > Prediction > Activation > Linear output > Weight.
- Without *Chain rule*, it's brute force recompute the entire network.

```math
\frac{dL}{dw} = \frac{dL}{dy}\frac{dy}{dz}\frac{dz}{dx}
```

In [7]:
import numpy as np

def g(x):
    return x**2

def f(u):
    return np.sin(u)

def composed(x):
    return f(g(x))

x = 1.5

u = g(x)

dy_du = np.cos(u)

du_dx = 2 * x

gradient = dy_du * du_dx

print(gradient)

-1.8845208681682175


In [8]:
eps = 1e-6

numerical = (
    composed(x + eps)
    - composed(x - eps)
) / (2 * eps)

print("Chain Rule:", gradient)
print("Numerical :", numerical)

Chain Rule: -1.8845208681682175
Numerical : -1.884520868022932


In [9]:
# Torch's chain-rule 

import torch

x = torch.tensor(1.5, requires_grad=True)
y = torch.exp(torch.sin(x**2))

y.backward()
print(x.grad)

tensor(-4.1031)


## 5. Backpropagation as Reverse Information Flow

- Like *Dynamic Programming*: next derivative requires previous ones.
- The flow is $\frac{\partial L}{\partial x}$ := how sensitive is the final loss to variable $x$.
- To solve the chain, we need the hidden layers -> *Backward*.
- Backprop starts at the end, where:

```math
\frac{\partial L}{\partial L} = 1
```

Then we keep going:

```math
\frac{\partial L}{\partial v} = \frac{\partial L}{\partial L}\frac{\partial L}{\partial v}
```

Branching:

```
        x
       / \
      /   \
     y     z
      \   /
       Loss
```
, chain rule becomes:

```math
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y}\frac{\partial y}{\partial x} + \frac{\partial L}{\partial z}\frac{\partial z}{\partial x}
```

- Automatic diferentiation.

Every node computes:

```math
\text{Incoming Gradient} \times \text{Local Derivative}
```

- Propagation = transfer prev gradient into the current node.

In [11]:
import numpy as np

class Node:

    def __init__(self, value):
        self.value = value
        self.grad = 0.0

x = Node(2.0)
u = Node(x.value **2)
L = Node(np.sin(u.value))

# Start with the loss
L.grad = 1.0
u.grad = L.grad * np.cos(u.value)
x.grad = u.grad * (2 * x.value)

print(x.grad)

-2.6145744834544478


In [12]:
# verification

def f(x):
    return np.sin(x**2)

eps = 1e-6
numerical = (f(2 + eps) - f(2 - eps)) / (2*eps)
print(numerical)

-2.614574483528198


## 6. Deriving Backprop for One Neuron

> How exactly one neuron changes its weights?
> Which direction to move $w$ and $b$?

- During forward, each neuron computes and remembers:
  - $x$: input
  - $z$: pre-activation
  - $a$: activation
  - $L$: loss

Suppose:

```math
z = wx + b, a = \sigma(z) = \frac{1}{1 + e^{-z}}, L = \frac{1}{2}(a - y)^2
```

```math
\frac{\partial L}{\partial a} = a - y, \frac{\partial a}{\partial z} = a(1 - a)
```

Delta ($\delta$)

```math
\frac{\partial L}{\partial z} = (a - y)a(1 - a)
```

then gradients:

```math
\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a} \frac{\partial a}{\partial z} \frac{\partial z}{\partial w} = (a-y)a(1-a)x
```

For bias, since $\frac{\partial z}{\partial b} = 1$ then:

```math
\frac{\partial L}{\partial b} = \frac{\partial L}{\partial z}
```

- Bias gradient is the neuron's delta $\delta$.

Summary:

```math
\delta = \frac{\partial L}{\partial z} = (a - y)a(1-a) \\
\frac{\partial L}{\partial w} = \delta x, 
\frac{\partial L}{\partial b} = \delta, 
\frac{\partial L}{\partial x} = \delta w
```

## 7. Backprop Through a Linear Layer

- 1000 inputs, 512 outputs, then $W$ contains:

```math
1000 \times 512 = 512,000 \text{ parameters}
```

- Calculus, matrix

Input $d$-dim:

```math
x = \begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_d \end{bmatrix}
```

Weights of $m$ neurons:

```math
W = \begin{bmatrix}
| & | & \cdots & | \\
w_1 & w_2 & \cdots & w_m \\
| & | & \cdots & |
\end{bmatrix}
```

then:

```math
z = x^TW + b
```

- Forward: every input feature contributes to every output.
- Backprop: every output contributes gradient back to every input.

**Dimensions**:

- N := Batch size
- D := Input features
- M := Output neurons

Then:

```math
X: (N, D), W: (D, M), b: (M), Z: (N, M)
```

- **Upstreaam gradient** $G = \frac{\partial L}{\partial Z}$ with shape $(N, M)$: gradient from the next layer. Then to derive $\frac{\partial L}{\partial W}$ in the current layer, we have:

```math
\text{forward: } z_{ij} = \sum_k x_{ik}w_{kj} + b_j \\
```

then any $w_{pq}$ depends on output $z_{iq}$, then:

```math
\frac{\partial z_{iq}}{\partial w_{pq}} = \begin{cases}
x_{ip}, & i = p \\
0, & i \ne p
\end{cases}
```

Chain rule:

```math
\frac{\partial L}{\partial w_{pq}} = \sum_i \frac{\partial L}{\partial z_{iq}}\frac{\partial z_{iq}}{\partial w_{pq}}
```

Then:

```math
\frac{\partial L}{\partial W} = X^TG (D, M)
```

Now wrt $X$:

```math
\frac{\partial L}{\partial x_{ip}} = \sum_j\frac{\partial L}{\partial z_{ij}}\frac{\partial z_{ij}}{\partial x_{ip}}, \text{ but } z_{ij} = \sum_k x_{ik}w_{kj} \\
\Rightarrow \frac{\partial z_{ij}}{\partial x_{ip}} = w_{pj} \\
\Rightarrow \frac{\partial L}{\partial X} = GW^T (N, D)
```

wrt $b$:

```math
\frac{\partial z_{ij}}{\partial b_{j}} = 1 \\
\Rightarrow \frac{\partial L}{\partial b_j} = \sum_iG_{ij} \\
\text{or } \frac{\partial L}{\partial b} = \sum_{\text{batch}}G
```

Summary:

- `dW`: how each param contributed to the error.
- `db`: accumulate error across all samples.
- `dX`: propagate the error signal to earlier layers.

In [ ]:
class Linear:

    def __init__():
        ...

    def forward():
        ...

    def backward():
        ...

## 8. Backprop Through Activation Functions

- Gradients flow through non-linear functions (activation).
- In backprop we receive $dA = \frac{\partial L}{\partial A}$, assume $A = f(Z)$, then:

```math
dZ = \frac{\partial L}{\partial Z} = dA \odot f'(Z) = J^TdA
```

where $\odot$ = Hadamard product, because $z_i$ produces $a_i$ directly without mat mul. And it's exactly Jacobian matrix:

```math
J = \begin{bmatrix}
f'(z_1) & 0 & 0 \\
0 & f'(z_2) & 0 \\
0 & 0 & f'(z_3)
\end{bmatrix}
```

### ReLU

```math
ReLU(z) = \max(0, z)
```

for $z = 0$, derivative is `undefined`.

```math
ReLU'(z) = \begin{cases}
1, & z > 0 \\
0, & z < 0
\end{cases}
```

so $dZ = dA \odot (Z > 0)$ with $(Z > 0)$ is binary mask.

### Sigmoid

```math
\sigma(z) = \frac{1}{1 + e^{-z}}
```

We have:

```math
\frac{d\sigma}{dz} = \frac{e^{-z}}{(1 + e^{-z})^2} = \sigma'(1 - \sigma') \\
\Leftrightarrow dZ = dA \odot A(1 - A)
```

### Tanh

```math
\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}
```

Then:

```math
\tanh'(z) = 1 - \tanh^2(z) \\
\Rightarrow dZ = dA \odot (1 - A^2)
```

### Summary

- When z too large -> Sigmoid saturates z near 0 or 1, Tanh saturates z near -1 or 1. In both cases, $f'(z) = 0$.
- ReLU avoids saturation for positive inputs: $ReLU'(z) = 1$.
- Linear: matmul, O(NDM) operations.
- Activation: Hadamard, O(NM) operations.

In [ ]:
class ReLU:
    ...

class Sigmoid:
    ...

class Tanh:
    ...

## 9. Building Reverse-Mode Automatic Differentiation

All of gradients (manually) we've learned for math ops, activations is how people learned DL in the 80s.

Today, we do **automatic differentiation (autograd)**.

- Each node cares 2 things:
  - compute output
  - pass gradients to its parents.


General Algo:

1. Store foward value
2. Store references to its parent nodes
3. Store a function that computes local derivatives
4. Traverse the graph in reverse topological order
5. Accumulate gradients into each parent


In [ ]:
import math

class Value:

    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)         # numerical value
        self.grad = 0.0                 # accumulated gradient
        self._prev = set(_children)     # parent nodes
        self._op = _op                  # the operation that created it
        self._backward = labmda: None   # how to backprop gradients to parents

    def __add__(self, other):
        ...

    def __mul__(self, other):
        ...

    def relu(self):
        ...

    def backward(self):
        """
        DFS - topo sort
        """
        ...

## 10. Implementing Backprop from Scratch (NumPy)

- To avoid high memory used when storing everything in forward:
  - gradient checkpointing
  - activation recomputation
  - reversitble networks
- If 10 bil params -> huge. -> **Reverse-mode** automatic differentiation

In [ ]:
import numpy as np
from .linear import Linear
from .relu import ReLU
from .mseloss import MSELoss


class NeuralNetwork:

    def __init__(self):
        self.fc1 = Linear(2, 4)
        self.relu = ReLU()
        self.fc2 = Linear(4, 1)
        self.loss = MSELoss()

    def forward(self, X, y):
        h = self.fc1.forward(X)
        h = self.relu.forward(h)
        pred = self.fc2.forward(h)
        loss = self.loss.forward(pred, y)
        return pred, loss

    def backward(self):
        grad = self.loss.backward()
        grad = self.fc2.backward(grad)
        grad = self.relu.backward(grad)
        grad = self.fc1.backward(grad)

In [ ]:
# Gradient Descent

learning_rate = 0.01

for layer in [net.fc1, net.fc2]:

    layer.W -= learning_rate * layer.dW
    layer.b -= learning_rate * layer.db

In [ ]:
# Complete training loop
net = NeuralNetwork()
learning_rate = 0.01

for epoch in range(1000):
    pred, loss = net.forward(X, y)
    net.backward()

    for layer in [net.fc1, net.fc2]:
        layer.W -= learning_rate * layer.dW
        layer.b -= learning_rate * layer.db

    if epoch % 100 == 0:
        print(epoch, loss)

In [ ]:
# torch
import torch

model = torch.nn.Sequential(
    torch.nn.Linear(2, 4),
    torch.nn.ReLU(),
    torch.nn.Linear(4, 1)
)

## 11. Gradient Checking

> How to know if gradient is correct?

- Check by move tiny amount, how much output changes.
- 2 types:
  - Analytical gradient: fast, inaccurate
  - Numerical gradient: Slow, simple
- Check if: Analytical gradient $\approx$ Numerical gradient

**Forward difference approximation**:

```math
f'(x) = \lim_{h \rightarrow 0} \frac{f(x + h) - f(x)}{h}
```

This is not accurate, say $f(x) = x^2 \Rightarrow f'(x) = 2x$. Then if $x = 3$, true $f(x) = 6$. But for the above formula: $f'(x) = 6 + h$. There's always error $h$.

**Central difference** is better, goes to both direction:

```math
f'(x) = \frac{f(x + h) - f(x - h)}{2h}
```

because the first-order errors cancel, the rest proportional to $h^2$ instead of $h$ (prove it by *Taylor Series*).

```math
f'(3) = \frac{(3 + h)^2 - (3 - h)^2}{2h} = 6
```

### Gradient checking in nn

- if backprop gives $g_{\text{analytic}}$ then:

```math
g_{\text{numeric}} = \frac{L(W_{ij} + \epsilon) - L(W_{ij} - \epsilon)}{2\epsilon}
```

### Relative error

- Never use absolute error

```math
\text{Relative Error } = \frac{|g_a - g_n|}{|g_a| + |g_n| + \epsilon}
```

where $\epsilon = 10^{-13}$.

- Threshold:
  - $< 10^{-7}$: Excellent
  - $10^{-7}-10^{-5}$: Usually acceptable
  - $10^{-5}$: Suspicious
  - $> 10^{-4}$: Likely a bug
  - $> 10^{-2}$: Almost certainly wrong

### Implementation

```
NUMERICAL-GRADIENT(network, X, y, parameter, eps=1e-5)
    for weight
        Increase slightly
        Measure loss
        Decrease slightly
        Measure loss
        Estimate slope
```

In [ ]:
# Use these to compare
import numpy as np

difference = np.linalg.norm(
    analytic - numeric
)

relative_error = (
    np.linalg.norm(analytic - numeric)
    /
    (
        np.linalg.norm(analytic)
        +
        np.linalg.norm(numeric)
        +
        1e-12
    )
)

print(relative_error)
print(difference)

## 12. Why Backprop Works So Efficiently

- Billions of params
- Function $L = f(\theta)$ where $\theta = (\theta_1, \theta_2,...,\theta_n)$. We can't perturb everyone of them slightly, compute loss again, then estimate the derivative.
- Have to reuse shared computations.
- Caching

> Dynamic Programming: Solve subproblem once -> reuse (forward: value, backward: gradient)

- Complexity: forward = backward = O(C) -> Total O(2C), with C = # params.
- So if 10 bil. samples, 7 bil. params -> the cost is not 10 bil. x 7 bil. It is # forward's FLOPs + # backward's FLOPs.

> **Floating Point Operations (FLOPs)**: Total number of mathematical calculations (add, mul) a NN performs to process one input.

| Outputs | Inputs | Best Method |
|---------|--------|-------------|
| 1 | Millions | Reverse-mode |
| Millions | 1 | Forward-mode |

- NN almost alwasy produce 1 scalar loss.

## 13. Common Gradient Pathologies (Bridge to the Next Modules)

What can be wrong? What assumptions break?

- After a series of multiplication of gradients, from Loss > Layer L > Layer L - 1 > ... > Layer 1 > Params. The figure might be:
  - extremely tiny
  - extremely huge
  - highly unstable
  - noisy
  - biased
- Compound interest
- Pathologies:
  - **Vanishing**: gradient = 0 -> Earlier layers stop learning. Symptoms:
    - training very slow
    - early layers never improve
    - network behaves like a shallow netwok
  - **Exploding**: gradient = millions -> Param updates become enomous.
    - training diverges
    - loss becomes NaN
  - **Dead neurons**: mainly in ReLU, gradient is always 0 -> Neron never learns again.
  - **Saturated activations**: in Sigmoid, if large input -> sigmoid  = +/- 1 -> gradient = 0 ($\sigma'(x)=\sigma(x)(1 - \sigma(x))$)
  - **Chaotic gradient flow**: different layers: gradient = 1, 100, .001, 50, .2... -> difficult to optimise.

In [1]:
# Random Jacobians

def random_gradient(depth):
    grad = 1.0
    history = []

    for _ in range(depth):
        derivative = np.random.normal(1,0.2)
        grad *= derivative
        history.append(grad)

    return np.array(history)